## PACOTES 

In [1]:
import time
import itertools
import numpy as np
import pandas as pd

from joblib import Parallel, delayed

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_curve,
    auc,
    matthews_corrcoef,
    log_loss
)

from scipy.stats import ks_2samp

## CONFIGURACOES

In [2]:
# CONFIG
inicio = time.time()

TARGET_COL = "status_fraude"
THRESHOLD = 0.50
estado_randomico = 42

numero_de_componentes = 2
inicializacoes_gausianas = 3
tipo_matriz_covariancia = "full"
erro_numerico = 1e-6

N_JOBS = 7

NOME_CSV = "2x2_visu_scores.csv"

## CRIACAO DO CSV

In [3]:
# LOAD DATASET
df = pd.read_csv("creditcard.csv")

# Se precisar renomear:
# df.rename(columns={"Class": TARGET_COL}, inplace=True)

features = [
    col for col in df.columns
    if col != TARGET_COL
]

combinacoes = list(itertools.combinations(features, 2))

print(f"Total de duplas: {len(combinacoes)}")


# FUNÇÃO SCORE FINAL
def calcular_score_final(auc_pr, mcc, ks, ll):

    auc_pr_norm = np.clip(auc_pr, 0, 1)

    mcc_norm = (mcc + 1) / 2
    mcc_norm = np.clip(mcc_norm, 0, 1)

    ks_norm = np.clip(ks, 0, 1)

    log_loss_norm = 1 / (1 + ll)

    score_final = (
        auc_pr_norm +
        mcc_norm +
        ks_norm +
        log_loss_norm
    ) / 4

    return round(float(score_final), 6)


# FUNÇÃO PARA PROCESSAR CADA DUPLA
def processar_dupla(f1, f2):

    try:
        temp = df[[f1, f2, TARGET_COL]].dropna()

        if temp.empty:
            return None

        X = temp[[f1, f2]]
        y_real = temp[TARGET_COL]

        if y_real.nunique() < 2:
            return None

        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        inicio_gmm = time.perf_counter()

        gmm = GaussianMixture(
            n_components=numero_de_componentes,
            covariance_type=tipo_matriz_covariancia,
            random_state=estado_randomico,
            reg_covar=erro_numerico,
            n_init=inicializacoes_gausianas
        )

        gmm.fit(X_scaled)

        fim_gmm = time.perf_counter()

        clusters = gmm.predict(X_scaled)

        ct = pd.crosstab(clusters, y_real)

        if 1 not in ct.columns:
            return None

        cluster_fraude = ct[1].idxmax()

        probabilidades = gmm.predict_proba(
            X_scaled
        )[:, cluster_fraude]

        probabilidades = np.clip(
            probabilidades,
            1e-15,
            1 - 1e-15
        )

        y_pred = (
            probabilidades >= THRESHOLD
        ).astype(int)

        precision_vals, recall_vals, _ = precision_recall_curve(
            y_real,
            probabilidades
        )

        auc_pr = auc(recall_vals, precision_vals)

        mcc = matthews_corrcoef(
            y_real,
            y_pred
        )

        ks = ks_2samp(
            probabilidades[y_real == 0],
            probabilidades[y_real == 1]
        ).statistic

        ll = log_loss(
            y_real,
            probabilidades
        )

        score_final = calcular_score_final(
            auc_pr,
            mcc,
            ks,
            ll
        )

        return {
            "Combinacao": f"{f1} | {f2}",
            "Feature_1": f1,
            "Feature_2": f2,
            "AUC_PR": round(float(auc_pr), 6),
            "MCC": round(float(mcc), 6),
            "KS": round(float(ks), 6),
            "Log_Loss": round(float(ll), 6),
            "Score_Final": score_final,
            "Tempo": round(float(fim_gmm - inicio_gmm), 6)
        }

    except Exception as e:
        print(f"ERRO -> {f1} + {f2}: {e}")
        return None


# PARALELISMO
resultados = Parallel(
    n_jobs=N_JOBS,
    verbose=10
)(
    delayed(processar_dupla)(f1, f2)
    for f1, f2 in combinacoes
)


# REMOVE ERROS/NONE
resultados = [
    r for r in resultados
    if r is not None
]

# DATAFRAME FINAL
df_resultados = pd.DataFrame(resultados)

df_resultados = df_resultados.sort_values(
    by="Score_Final",
    ascending=False
).reset_index(drop=True)

df_resultados["Posicao_Rank"] = df_resultados.index + 1

display(df_resultados)

# SALVA CSV
df_resultados.to_csv(
    NOME_CSV,
    index=False
)

fim = time.time()

print("\nArquivo criado com sucesso:")
print(NOME_CSV)

print(
    f"Tempo total: "
    f"{fim - inicio:.4f} segundos finalizados"
)

Total de duplas: 435


[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.
[Parallel(n_jobs=7)]: Done   4 tasks      | elapsed:   25.8s
[Parallel(n_jobs=7)]: Done  11 tasks      | elapsed:  1.0min
[Parallel(n_jobs=7)]: Done  18 tasks      | elapsed:  1.7min
[Parallel(n_jobs=7)]: Done  27 tasks      | elapsed:  3.3min
[Parallel(n_jobs=7)]: Done  36 tasks      | elapsed:  4.7min
[Parallel(n_jobs=7)]: Done  47 tasks      | elapsed:  5.9min
[Parallel(n_jobs=7)]: Done  58 tasks      | elapsed:  7.3min
[Parallel(n_jobs=7)]: Done  71 tasks      | elapsed:  9.1min
[Parallel(n_jobs=7)]: Done  84 tasks      | elapsed: 10.7min
[Parallel(n_jobs=7)]: Done  99 tasks      | elapsed: 12.8min
[Parallel(n_jobs=7)]: Done 114 tasks      | elapsed: 14.7min
[Parallel(n_jobs=7)]: Done 131 tasks      | elapsed: 17.4min
[Parallel(n_jobs=7)]: Done 148 tasks      | elapsed: 19.3min
[Parallel(n_jobs=7)]: Done 167 tasks      | elapsed: 21.9min
[Parallel(n_jobs=7)]: Done 186 tasks      | elapsed: 24.6min
[Parallel(

,Combinacao,Feature_1,Feature_2,AUC_PR,MCC,KS,Log_Loss,Score_Final,Tempo,Posicao_Rank
0,V11 | V17,V11,V17,0.579100,0.245216,0.857038,0.121061,0.737689,126.248478,1
1,V15 | V17,V15,V17,0.533549,0.259166,0.793022,0.090869,0.718213,93.518625,2
2,V4 | V17,V4,V17,0.567597,0.176108,0.855490,0.166150,0.717166,44.123116,3
3,V3 | V17,V3,V17,0.488633,0.236428,0.844484,0.147757,0.705649,63.444243,4
4,V17 | V22,V17,V22,0.515056,0.223334,0.790095,0.126568,0.701117,49.800208,5
...,...,...,...,...,...,...,...,...,...,...
430,V13 | tempo_desde_a_primeira_transacao,V13,tempo_desde_a_primeira_transacao,0.002448,0.013613,0.168267,7.280005,0.199573,25.590390,431
431,V25 | tempo_desde_a_primeira_transacao,V25,tempo_desde_a_primeira_transacao,0.002303,0.011251,0.148871,6.834195,0.196111,10.149103,432
432,V4 | tempo_desde_a_primeira_transacao,V4,tempo_desde_a_primeira_transacao,0.002181,0.008853,0.132965,7.163906,0.190516,27.265349,433
433,V22 | tempo_desde_a_primeira_transacao,V22,tempo_desde_a_primeira_transacao,0.002385,0.009723,0.127227,7.086105,0.189536,28.867027,434



Arquivo criado com sucesso:
2x2_visu_scores.csv
Tempo total: 3260.9276 segundos finalizados
